# RAG Workbench — GPU corpus backfill (Google Colab, free T4)

Embeds the missing **2024+2025 annual (10-K/20-F)** and **1 year of 10-Q** filings into the live DuckDB corpus on a **free Colab GPU** (~30-60 min), then publishes to the HF dataset and restarts the Space. Reuses the repo's own embedding + chunking + guardrail code, so the new vectors match the existing corpus exactly (Qwen3-Embedding-0.6B, 1024-dim).

### One-time setup before running
**This repo is private**, so the Colab *GitHub* tab can't open it (404). Instead: **File → Upload notebook** and pick `notebooks/colab_backfill.ipynb` from your machine.

1. **Runtime → Change runtime type → T4 GPU** (free).
2. Click the **🔑 (key) icon** in the left sidebar → **Add new secret**, and add **two** secrets (toggle *Notebook access* on for each):
   - `HF_TOKEN` = your Hugging Face **write** token (uploads the dataset + restarts the Space).
   - `GH_TOKEN` = a GitHub token with **read** access to this private repo (used only to clone the code). A fine-grained token with *Contents: Read* on `evangoh122/Rag_workbench` is enough.
3. **Runtime → Run all.**

Safety: the guardrail refuses to upload if the DB shrinks, loses any filing, or has non-1024-dim / null vectors — so a bad run leaves prod untouched. The dataset keeps git history, so the prior DB is always one revert away.

In [ ]:
# 1) Verify GPU, clone the (private) repo with a token, install deps
import torch, os
assert torch.cuda.is_available(), \
    'No GPU! Runtime > Change runtime type > T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))

from google.colab import userdata
GH_TOKEN = userdata.get('GH_TOKEN')   # GitHub token with read access to the private repo
if not os.path.isdir('Rag_workbench'):
    !git clone -q https://{GH_TOKEN}@github.com/evangoh122/Rag_workbench.git
%cd Rag_workbench
!pip install -q -r requirements.txt

# Make sure the dependency install did not replace Colab's CUDA torch with a CPU build.
import importlib, torch; importlib.reload(torch)
assert torch.cuda.is_available(), \
    'CUDA torch was clobbered by pip. Re-run this cell, or tell Claude which torch version Colab has.'
print('CUDA OK after install:', torch.cuda.get_device_name(0))

In [ ]:
# 2) Secrets + environment (forces the exact corpus model; bigger batch for GPU)
import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['EDGAR_USER_AGENT'] = 'Evan Goh evangohsg@gmail.com'
os.environ['EMBEDDING_PROVIDER'] = 'sentence-transformers'
os.environ['ST_EMBEDDING_MODEL'] = 'Qwen/Qwen3-Embedding-0.6B'
os.environ['EMBEDDING_DIM'] = '1024'
os.environ['EMBED_DOC_BATCH'] = '128'   # GPU can take a much larger batch than the CPU runner
DATASET_REPO = 'egoh33/Rag-workbench'
SPACE_REPO   = 'egoh33/Auditable-Filing-QA'
print('Env set. Dataset:', DATASET_REPO)

In [ ]:
# 3) Download the CURRENT live DB from the dataset, then snapshot it (guardrail baseline)
import shutil
from huggingface_hub import hf_hub_download
os.makedirs('data', exist_ok=True)
path = hf_hub_download(DATASET_REPO, 'rag.duckdb', repo_type='dataset', token=os.environ['HF_TOKEN'])
shutil.copy(path, 'data/rag.duckdb')
print('Downloaded:', round(os.path.getsize('data/rag.duckdb')/1e6, 1), 'MB')
!python scripts/etl_guardrail.py snapshot data/rag.duckdb /tmp/before.json

In [ ]:
# 4) Embed on the GPU — annual (2024+2025) then 1 year of 10-Qs. This is the long cell (~30-60 min).
#    Already-stored accessions are skipped, so it's safe to re-run if interrupted.
!python scripts/embed_additional.py --forms annual
!python scripts/embed_additional.py --forms 10-Q --quarters 3

In [ ]:
# 5) GUARDRAIL — refuse to publish if anything was destroyed. Stops here on failure.
rc = os.system('python scripts/etl_guardrail.py verify data/rag.duckdb /tmp/before.json')
assert rc == 0, 'Guardrail FAILED — NOT uploading. Read the output above; prod is untouched.'
print('Guardrail passed.')

In [ ]:
# 6) Publish: upload the new DB to the dataset and restart the Space
from huggingface_hub import HfApi
api = HfApi(token=os.environ['HF_TOKEN'])
info = api.repo_info(DATASET_REPO, repo_type='dataset')
print('Pre-upload dataset commit (rollback target if needed):', info.sha)
api.upload_file(
    path_or_fileobj='data/rag.duckdb', path_in_repo='rag.duckdb',
    repo_id=DATASET_REPO, repo_type='dataset',
    commit_message='ETL(colab-gpu): backfill 2024+2025 annual + 1yr 10-Q',
)
api.restart_space(SPACE_REPO)
print('Uploaded + Space restarting. Backfill complete — tell Claude to sync the laptop DB.')